In [ ]:
# CONFIGURATION
# Available domains:
#   capital_markets, education, financial, healthcare, hr, insurance, iot,
#   manufacturing, marketing, real_estate, retail, supply_chain, telecom
DOMAIN      = "retail"

# Available sinks: lakehouse | warehouse | eventhouse | sql-database | sql-server
SINK_TYPE   = "lakehouse"

# Available sizes: small | medium | large | fabric_demo
SCALE_SIZE  = "small"

# Available modes: seeding | streaming
MODE        = "seeding"

SEED        = 42

# CONNECTION (fill in for your sink)
WORKSPACE_ID       = ""   # Fabric workspace GUID (sink=lakehouse)
LAKEHOUSE_ID       = ""   # Lakehouse GUID (sink=lakehouse)
SQL_CONN           = ""   # ODBC string (sink=warehouse|sql-database|sql-server)
EVENTHOUSE_URI     = ""   # KQL cluster URI (sink=eventhouse)
EVENTHOUSE_DB      = ""   # KQL database name (sink=eventhouse)


In [ ]:
import importlib
from sqllocks_spindle import Spindle
from sqllocks_spindle.cli import _get_domain_registry

registry = _get_domain_registry()
module_path, class_name, _ = registry[DOMAIN]
module = importlib.import_module(module_path)
domain_obj = getattr(module, class_name)(schema_mode="3nf")

spindle = Spindle()
print(f"Loaded domain: {DOMAIN}")


In [ ]:
if MODE == "seeding":
    result = spindle.generate(domain=domain_obj, scale=SCALE_SIZE, seed=SEED)
    tables = result.tables
    errors = result.verify_integrity()
    assert errors == [], f"FK integrity errors: {errors}"
elif MODE == "streaming":
    tables = {}
    for table_name, df in spindle.generate_stream(domain=domain_obj, scale=SCALE_SIZE, seed=SEED):
        tables[table_name] = df
        print(f"  Yielded: {table_name} ({len(df):,} rows)")
else:
    raise ValueError(f"Unknown MODE: {MODE}")

print(f"Total: {sum(len(df) for df in tables.values()):,} rows across {len(tables)} tables")


In [ ]:
# WRITE TO SINK
if SINK_TYPE == "lakehouse":
    from azure.identity import InteractiveBrowserCredential
    from deltalake import write_deltalake
    token = InteractiveBrowserCredential(tenant_id="2536810f-20e1-4911-a453-4409fd96db8a").get_token("https://storage.azure.com/.default").token
    opts = {"bearer_token": token, "use_emulator": "false"}
    for name, df in tables.items():
        path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/{DOMAIN}_{name}"
        write_deltalake(path, df, mode="overwrite", storage_options=opts, schema_mode="overwrite")
        print(f"  {DOMAIN}_{name}: {len(df):,} rows -> lakehouse")

elif SINK_TYPE in ("warehouse", "sql-database", "sql-server"):
    from sqllocks_spindle.fabric.sql_database_writer import FabricSqlDatabaseWriter
    auth = "sql" if SINK_TYPE == "sql-server" else "cli"
    writer = FabricSqlDatabaseWriter(SQL_CONN, auth_method=auth)
    # Wrap tables dict as a result-like object for writer.write()
    class _R:
        def __init__(self, t, go): self.tables=t; self.generation_order=go; self.schema=None
    pseudo = _R(tables, list(tables.keys()))
    writer.write(pseudo, schema_name="dbo", mode="create_insert")
    print(f"Written to {SINK_TYPE}")

elif SINK_TYPE == "eventhouse":
    print("Eventhouse: wire EVENTHOUSE_URI/DB to your KQL ingest client here")
    for name, df in tables.items():
        print(f"  Would ingest {name}: {len(df):,} rows")

print("Done")
